# Validación de modelos

Hemos construido un modelo. Pero, ¿qué tan bueno es?

En esta lección, aprenderemos a usar la validación del modelo para medir la calidad de nuestro modelo. Medir la calidad del modelo es la clave para mejorar nuestros modelos de manera iterativa.

## ¿Qué es la validación de modelos?

Siempre vamos a querer evaluar casi todos los modelos que construyamos. En la mayoría de las aplicaciones (aunque no en todas), la medida relevante de la calidad del modelo es la precisión predictiva. En otras palabras, ¿las predicciones del modelo estarán cerca de lo que realmente sucede?

Muchas personas cometen un gran error al medir la precisión predictiva. Hacen predicciones con sus datos de entrenamiento y comparan esas predicciones con los valores objetivo en los datos de entrenamiento. Veremos el problema con este enfoque y cómo solucionarlo en un momento, pero primero pensemos en cómo lo haríamos.

Primero necesitaríamos resumir la calidad del modelo de una manera comprensible. Si comparamos los valores predichos y reales de 10,000 casas, probablemente encontraremos una mezcla de buenas y malas predicciones. Revisar una lista de 10,000 valores predichos y reales sería inútil. Necesitamos resumir esto en una sola métrica.

Existen muchas métricas para resumir la calidad del modelo, pero comenzaremos con una llamada Error Absoluto Medio (Mean Absolute Error, también conocido como MAE). Desglosaremos esta métrica comenzando con la última palabra, error.

El error de predicción para cada casa es:

$$
error = real − predicción
$$

Entonces, si una casa costó 150,000 y predijiste que costaría 100,000, el error es 50,000.

Con la métrica MAE, tomamos el valor absoluto de cada error. Esto convierte cada error en un número positivo. Luego tomamos el promedio de esos errores absolutos. Esta es nuestra medida de la calidad del modelo. En lenguaje sencillo, se puede decir:

> En promedio, nuestras predicciones están equivocadas por aproximadamente X.

Para calcular el MAE, primero necesitamos un modelo.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

# Cargar datos
ruta = './data/melb_data.csv'
melbourne_data = pd.read_csv(ruta) 

# Quitar nulos
melbourne_data_filtrada = melbourne_data.dropna(axis=0)

# Elegir target
y = melbourne_data_filtrada.Price

# Elegir features
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']

# Crear X
X = melbourne_data_filtrada[melbourne_features]


# Definir modelo
# Fijamos la semilla del estimador para reproducir sus decisiones ante empates.
modelo = DecisionTreeRegressor(random_state=0)

# Ajustar modelo
modelo.fit(X, y)

print(f"Filas originales: {len(melbourne_data):,}; completas tras dropna: {len(X):,}")

Filas originales: 13,580; completas tras dropna: 6,196


Ya tenemos nuestro modelo y ahora podemos calcular el MAE. Para esto utilizaremos también `sklearn`.

In [2]:
from sklearn.metrics import mean_absolute_error

predicciones = modelo.predict(X)
mae_in_sample = mean_absolute_error(y, predicciones)
print(f"MAE in-sample (modelo ajustado y evaluado en todas las filas limpias): {mae_in_sample:,.6f}")

MAE in-sample (modelo ajustado y evaluado en todas las filas limpias): 434.715946


## El Problema con los puntajes "dentro de la muestra"

La medida que acabamos de calcular puede llamarse un puntaje "dentro de la muestra". Usamos una sola "muestra" de casas tanto para construir el modelo como para evaluarlo. Esto está mal. Veamos por qué.

Imaginemos que en el mercado inmobiliario el color de la puerta no está relacionado con el precio de la casa.

Sin embargo, en la muestra de datos que usamos para construir el modelo, todas las casas con puertas verdes eran muy caras. El trabajo del modelo es encontrar patrones que predigan los precios de las casas, así que verá este patrón y siempre predecirá precios altos para las casas con puertas verdes.

Dado que este patrón se derivó de los datos de entrenamiento, el modelo parecerá preciso en los datos de entrenamiento.

Pero si este patrón no se mantiene cuando el modelo ve nuevos datos, el modelo sería muy impreciso cuando se use en la práctica.

Dado que el valor práctico de los modelos proviene de hacer predicciones sobre nuevos datos, medimos el rendimiento en datos que no se usaron para construir el modelo. La forma más sencilla de hacer esto es excluir algunos datos del proceso de construcción del modelo y luego usarlos para probar la precisión del modelo en datos que no ha visto antes. Estos datos se llaman datos de validación.

## Partiendo los datos
Vamos a dividir nuestros datos en 2 subconjuntos: set de entrenamiento y set de pruebas.

`scikit-learn` tiene una función llamada `train_test_split` para dividir los datos en estas dos partes. Usaremos algunos de esos datos como datos de entrenamiento para ajustar el modelo, y usaremos los otros datos como datos de validación para calcular el error absoluto medio (`mean_absolute_error`).

In [3]:
from sklearn.model_selection import train_test_split

Dividimos los datos en datos de entrenamiento y de validación, tanto para las características como para el objetivo

Esta división se basa en un generador de números aleatorios. Por lo tanto, podemos darle un `random_state` para que siempre que corramos nuestro código, tengamamos exactamente los mismos sets de entrenamiento y de pruebas.

La función `train_test_split` recibe 2 argumentos y 1 opcional. El primer argumento es el dataframe que contiene los features, y el segundo argumento contiene los precios (targets). 

La función nos regresará 4 conjuntos de datos. 2 para entrenar y 2 para probar

In [4]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)

assert len(train_X) + len(val_X) == len(X)
assert train_X.index.intersection(val_X.index).empty
assert train_X.index.equals(train_y.index)
assert val_X.index.equals(val_y.index)
print(f"Entrenamiento: {len(train_X):,} filas ({len(train_X)/len(X):.0%})")
print(f"Validación: {len(val_X):,} filas ({len(val_X)/len(X):.0%})")
print("Comprobación: sin índices compartidos y objetivos alineados con las características.")

Entrenamiento: 4,647 filas (75%)
Validación: 1,549 filas (25%)
Comprobación: sin índices compartidos y objetivos alineados con las características.


In [5]:
train_X.head()

,Rooms,Bathroom,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude
10385,3,1.0,206.0,110.0,1980.0,-37.87107,145.04991
5805,2,1.0,0.0,73.0,2000.0,-37.85900,144.97670
8488,2,1.0,2701.0,79.0,2011.0,-37.81090,144.86840
6672,3,1.0,670.0,116.0,1940.0,-37.81340,144.87450
776,6,3.0,708.0,275.0,1988.0,-37.91810,145.04400


In [6]:
train_y.head()

10385    1060000.0
5805      390000.0
8488      502000.0
6672     1055000.0
776      1900000.0
Name: Price, dtype: float64

In [7]:
val_X.head()

,Rooms,Bathroom,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude
4850,2,1.0,96.0,71.0,1880.0,-37.85010,144.99530
2307,2,1.0,0.0,70.0,1965.0,-37.89020,144.99070
10090,2,1.0,136.0,58.0,1892.0,-37.85542,144.99571
3645,3,2.0,205.0,184.0,1995.0,-37.79930,145.02670
4930,2,1.0,400.0,88.0,1955.0,-37.73520,144.98520


In [8]:
val_y.head()

4850      815000.0
2307      655000.0
10090     957500.0
3645     1330000.0
4930      722000.0
Name: Price, dtype: float64

Ahora definamos nuestro modelo nuevamento y ajustémoslo

In [9]:
# La misma semilla permite repetir el ajuste del árbol.
modelo = DecisionTreeRegressor(random_state=0)

# Ajustar únicamente con datos de entrenamiento.
modelo.fit(train_X, train_y)
train_predictions = modelo.predict(train_X)
val_predictions = modelo.predict(val_X)
mae_entrenamiento = mean_absolute_error(train_y, train_predictions)
mae_validacion = mean_absolute_error(val_y, val_predictions)
print(f"MAE entrenamiento (75% usado para ajustar): {mae_entrenamiento:,.6f}")
print(f"MAE validación (25% no usado para ajustar): {mae_validacion:,.6f}")
print(f"Precio medio en validación: {val_y.mean():,.2f}")
print(f"MAE validación / precio medio: {mae_validacion / val_y.mean():.2%}")

MAE entrenamiento (75% usado para ajustar): 281.471917
MAE validación (25% no usado para ajustar): 265,345.351194
Precio medio en validación: 1,091,516.64
MAE validación / precio medio: 24.31%


**¡¿Ves la diferencia?!**

El primer árbol, ajustado y evaluado sobre las 6,196 filas completas, obtuvo un MAE in-sample de **434.715946**. Después de dividir los datos, el segundo árbol obtuvo un MAE de **281.471917 en las 4,647 filas de entrenamiento** y **265,345.351194 en las 1,549 filas de validación** que no se usaron para ajustarlo.

El aumento del error en validación muestra por qué un resultado muy bajo en entrenamiento no garantiza buenas predicciones en viviendas nuevas. Como referencia, el precio medio en validación es **1,091,516.64**; el MAE de validación equivale al **24.31%** de ese promedio. Este cociente aporta contexto, no es el promedio de los errores porcentuales de cada vivienda ni determina por sí solo la utilidad práctica del modelo.

Hay muchas maneras de mejorar este modelo, como experimentar para encontrar mejores características o diferentes tipos de modelos.

## Respuestas y resultados de esta ejecución

1. **¿Qué mide el MAE y qué tan bueno es el modelo?** El MAE es el promedio de $|y_i-\hat y_i|$: expresa la magnitud media del error en las mismas unidades monetarias de `Price`. Un valor menor es mejor para esta métrica. En las 1,549 viviendas de validación se obtuvo **265,345.351194**, equivalente al **24.31%** del precio medio (1,091,516.64); este cociente aporta contexto, no es un porcentaje de error calculado vivienda por vivienda ni basta para decidir utilidad práctica.

2. **¿Por qué es problemático evaluar in-sample?** El primer árbol se ajustó y evaluó sobre las 6,196 filas completas: su MAE fue **434.715946**. El segundo se ajustó solo sobre entrenamiento: su MAE en esas 4,647 filas fue **281.471917**. Ambos errores de entrenamiento son muy bajos frente al MAE de validación; pueden reflejar memorización y no estiman por sí solos el error en viviendas nuevas.

3. **¿Cómo se dividieron los datos?** Se mantuvo `dropna(axis=0)`: quedaron 6,196 de 13,580 filas, usando `Price` como objetivo y las siete características originales. `train_test_split(X, y, random_state=0)` separó **4,647 filas de entrenamiento (75%)** y **1,549 de validación (25%)**. Las comprobaciones ejecutadas verifican índices disjuntos, objetivos alineados y que ambas partes suman todas las filas limpias. El modelo de validación se entrenó solo con `train_X` y `train_y`.

4. **¿Ves la diferencia?** Sí: al evaluar en viviendas reservadas, el error sube a **265,345.351194**. El texto explicativo y las salidas anteriores reflejan los resultados de esta ejecución. Se fijó también `random_state=0` en ambos estimadores para hacer reproducibles los desempates del árbol. Esta evaluación usa una sola partición; no es validación cruzada.

**Alcance:** el notebook 2 reutiliza la misma partición para seleccionar un hiperparámetro. Por eso ese MAE de validación no debe presentarse como una evaluación final en un conjunto de prueba independiente. Además, los resultados se refieren a las filas completas conservadas, no necesariamente a toda la población original.

**Entorno utilizado:** Python 3.12.14, pandas 3.1.0, NumPy 2.5.4, scikit-learn 1.9.1, Matplotlib 3.11.2, Seaborn 0.13.2. No se cambiaron el CSV, las imágenes ni las siete variables originales. Las celdas `head()` muestran los cuatro subconjuntos y la celda final vacía original se conserva.
